# 2D3V 网格基础测试

逐个检查 `PeriodicGrid` 的构造、`size`、`spacing`、`wrap_index`、`index` 和 `wrap_position`。网格只在 x-y 平面变化，z 坐标保留原值。

In [ ]:
import os
import sys
sys.path.insert(0, os.environ['BORIS_MODULE_DIR'])
import boris_cpp as boris

origin = boris.Vec3(1.0, 2.0, 7.0)
extent = boris.Vec3(4.0, 6.0, 0.0)
grid = boris.PeriodicGrid(origin, extent, (4, 3, 8))
assert (grid.origin.x, grid.origin.y, grid.origin.z) == (1.0, 2.0, 7.0)
assert (grid.extent.x, grid.extent.y, grid.extent.z) == (4.0, 6.0, 0.0)

# 2D 网格只有一个 z 平面，总节点数是 4 * 3。
assert tuple(grid.cells) == (4, 3, 1)
assert grid.size() == 12

# x 方向间距是 4/4=1，y 方向间距是 6/3=2；z 没有网格间距。
dx = grid.spacing()
assert (dx.x, dx.y, dx.z) == (1.0, 2.0, 0.0)

# 周期索引把负数也绕回有效范围。
assert grid.wrap_index(-1, 4) == 3
assert grid.wrap_index(4, 4) == 0

# index(-1, 3, 8) 等同于 index(3, 0, 0)，线性位置为 3*3+0。
assert grid.index(-1, 3, 8) == 9

# x=-1 绕回 3，y=9 绕回 3，z=123 保持不变。
point = grid.wrap_position(boris.Vec3(-1.0, 9.0, 123.0))
assert (point.x, point.y, point.z) == (3.0, 3.0, 123.0)

print('PeriodicGrid 的所有函数都通过了。')

# 2D3V 粒子可以有 z 速度，但不会沿 z 移动。
mesh = boris.UniformMesh(
    boris.Vec3(0, 0, 0), boris.Vec3(4, 4, 0), (4, 4, 1),
    boris.Vec3(0, 0, 1), boris.Vec3(0, 0, 0),
)
particle = boris.Particle(boris.Vec3(1, 1, 9), boris.Vec3(0, 0, 0), 1, 1)
boris.push(particle, 0.1, mesh)
assert abs(particle.velocity.z - 0.1) < 1e-12
assert particle.position.z == 9
print('粒子 z 速度会更新，但 z 位置保持不变。')